# Collect a small offline-training dataset

Run random steps in FrozenLake using **mouse-env**, save them into a `Datastore`, and push the dataset to the Hugging Face Hub.

This dataset is intentionally discrete-action only so `02_train_offline.ipynb` can load it and train the DQN example directly.


In [1]:
from mouse_envs import EnvConfig, make_vector_env
from mouse_core.data import Datastore, push_stores_to_hub

# Hub dataset repo name. A bare name is created under your authenticated HF user.
DATASET_ID = "mouse-example-dataset"

# Dataset subset/config used to label this particular collection run in the repo.
DATASET_CONFIG = "frozenlake"

# FrozenLake has four discrete actions: left, down, right, up.
MAX_ACTIONS = 4

# Number of random environment transitions to collect and upload.
STEPS_PER_ENV = 200


## The environments

We describe each environment with an `EnvConfig`.
mouse-env builds it, handles resets, and gives back clean step results.


In [2]:
# FrozenLake has four discrete actions, matching the DQN head in the training notebook.
ENV_CONFIG = EnvConfig(
    "FrozenLake-v1",
    seed=0,
    num_envs=1,
    max_episode_steps=100,
    kwargs={"is_slippery": True},
)

## Collect a few steps

Call `step(actions)`. Get back results.
Attach the action we just used, put it first, and append the row.

In [3]:
def result_to_row(result, action):
    return {"action": action["action"], **result}


def collect_steps(store, cfg, num_steps):
    """Run random steps and save (action, result) rows."""
    env = make_vector_env(cfg)
    try:
        for _ in range(num_steps):
            actions = env.sample_random_actions()
            results, _ = env.step(actions)
            row = result_to_row(results[0], actions[0])
            row["env"] = cfg.group_id
            store.append(row)
    finally:
        env.close()
    return num_steps


store = Datastore()
n_steps = collect_steps(store, ENV_CONFIG, STEPS_PER_ENV)
print(f"{ENV_CONFIG.group_id}: {n_steps} steps")
print(store)

FrozenLake-v1: 200 steps
Datastore(steps=200)


## Push to the Hugging Face Hub

Push the collected rows under the hard-coded dataset name. The Hub client creates or updates the dataset under your authenticated Hugging Face account.


In [4]:
# config_name creates a "bin" inside the dataset repo.
# push deletes the old README so a fresh one is written.
url = push_stores_to_hub(
    [store],
    repo_id=DATASET_ID,
    split="train",
    config_name=DATASET_CONFIG,
    private=False,
)
print(f"Pushed dataset to {url}")


Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushed to micahr234/mouse-example-dataset (train: 200 steps)
Pushed dataset to https://huggingface.co/datasets/micahr234/mouse-example-dataset
